# Clustering the result of a MCA with DBSCAN

#### Documentation 

[Comparing clustering algorithms](https://hdbscan.readthedocs.io/en/latest/comparing_clustering_algorithms.html) 


In [ ]:
import numpy as np
import pandas as pd

import sqlite3 as sql

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import seaborn as sns

import networkx as nx
import itertools

import scipy.cluster.hierarchy as sch
from scipy.cluster.hierarchy import fcluster
from scipy.spatial.distance import cdist

import fanalysis.ca as fa 

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.cluster import DBSCAN

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors

from prince import MCA


In [ ]:
### this library allows to execute SQL queries on dataframes
# and even joins
import duckdb

In [ ]:
### Importer un module de fonctions crées ad hoc
##  ATTENTION : le fichier 'sparql_functions.py' doit se trouver 
#   dans un dossier qui se situe dans le chemin ('path') de recherche
#   vu par le présent carnet Jupyter afin que
#   l'importation fonctionne correctement

import sys
from importlib import reload


# Add parent directory to the path
sys.path.insert(0, '..')

### If you want to add the parent-parent directory,
sys.path.insert(0, '../..')



In [ ]:
## these are local libraries of functions 
import bivariate_library as bl
import correspondence_analysis_library as cal
import cluster_functions as cf


In [ ]:
### Use this command to reload the functions 
# if you modify them
print(reload(cf))


In [ ]:
import warnings
warnings.filterwarnings('ignore')


## Create a dataframe with the data to be analysed

In this notebook, we use the data produced in the [multiple correspondence analysis (MCA) notebook ](da5_MCA.ipynb), which was prepared and exported into this file [this file](../da_data/da5-MCA-clusters.csv) after the sparse data was aggregated.




In [ ]:
file_address='../da_data/da5-MCA-clusters.csv'
df_pm = pd.read_csv(file_address)
df_pm.head(3)

In [ ]:
print(df_pm.columns.to_list())

In [ ]:
### Rename columns to have shorter labels
df_pm.columns=['person_uri',
 'labelPer',
 'birthYear',
 'gender',
 'labelPlace',
 'geometry',
 'REGION',
 'NAME_ENGL',
 'country',
 'per_activ',
 'pk_person_features',
 'occup_main',
 'occup_sec',
 'empl']

In [ ]:
### Inspect the dataframe and 
# notably if there are missing values
df_pm.info()

In [ ]:
### change display options for jupyter notebooks
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# Reset to default settings if needed later
# pd.reset_option('display.max_columns')

## Factor analysis (MCA)

We only consider for MCA the coded features of persons, transformed into qualitative categories 

We don't use the periods for defining profiles, our research question is about inspecting the correlation between profiles and periods, and identify specific profiles for specific periods

In [ ]:
### Define the variables that will be used for analysing the population profiles
df_actives = df_pm[['gender', 'country', 'occup_sec', 'empl']].copy(deep=True)
print(df_actives.columns)
df_actives.head(2)

In [ ]:
### We add here also the main occupation, hoping it helps to get more structure
df_actives = df_pm[['gender', 'country', 'occup_main', 'occup_sec', 'empl']].copy(deep=True)
print(df_actives.columns)
df_actives.head(2)

### Process in two steps

First fit, then transform.

The mca variable will be used to project illustrative variables, in our case periods of activity

In [ ]:
### MCA with benzecri correction: whole intertia in 5 axis
# Equivalent to fit_transform but in to steps
mca = MCA(n_components=3, n_iter=10, correction='benzecri',random_state=42)
mca.fit(df_actives)

In [ ]:
### Benzecri correction and approximation allows to summarize 
# the most relevant aspects of the variance in five axes (in this case)
mca.eigenvalues_summary

In [ ]:
### Coordinates of categories, i.e. columns in the complete disjoint table
print(len(mca.column_coordinates(df_actives)))
dfcc = mca.column_coordinates(df_actives)
dfcc.head()

In [ ]:
### Coordinates of persons (rows)
# The same result as using the method 'fit_transform'
print(len(mca.row_coordinates(df_actives)))
dfrc = mca.row_coordinates(df_actives)
dfrc.head()

In [ ]:
### Equivalent to fit transform in one step
X_mca=mca.transform(df_actives)
X_mca.head()

### Prepare first two axis

In [ ]:
ax1=mca.eigenvalues_summary.iloc[0, 1]
ax2=mca.eigenvalues_summary.iloc[1, 1]

### Calculate position of illustrative variable 'per_activ'

We don't use the periods for defining profiles, our research question is about inspecting the correlation between profiles and periods, and identify specific profiles for specific periods

In [ ]:
df_illustrative = df_pm[['per_activ']].copy(deep=True)
# df_illustrative.head(2)

In [ ]:
df_per_activ=pd.DataFrame(df_pm.groupby('per_activ').size())
# df_per_activ.index=df_per_activ.index.map(lambda x: f'per_activ__{x}')
df_per_activ.columns=['number']
df_per_activ

In [ ]:
### This script situates the illustrative variable 'per_activ'
# at the mean position in the geometric space using the coordinates 
# of all the individuals

row_coords = mca.row_coordinates(df_actives)

# 3. Select the single illustrative variable column
illus_var = df_illustrative['per_activ'] # Replace with actual column name

# 4. Calculate position for each category in that variable
supp_positions = {}

for category in illus_var.dropna().unique():
    # Find indices of individuals who have this category
    mask = illus_var == category
    
    # Calculate the barycenter (mean) of their row coordinates
    # This is the position of the supplementary category
    centroid = row_coords[mask].mean(axis=0)
    
    supp_positions[category] = centroid.values

# 5. Convert to DataFrame for easy use/plotting
df_per_activ_pos = pd.DataFrame(
    supp_positions, 
    index=[i for i in range(row_coords.shape[1])]
).T  # Transpose so rows are categories, columns are dimensions

df_per_activ_pos.head(2)

In [ ]:
### We add the label and the number of individuals to the mean position
df_per_activ=df_per_activ.join(df_per_activ_pos)
df_per_activ

In [ ]:
### dimensions
d1=1 # 0
d2=2 # 1

# Create the plot
fig, ax = plt.subplots(figsize=(20, 20))

# Plot Active Variables (Blue/Grey default)
ax.scatter(dfcc[d1], dfcc[d2], s=30, alpha=0.7, c='blue', label='Active Variables', zorder=3)

# Label Active Variables
for label, x, y in zip(dfcc.index, dfcc[d1], dfcc[d2]):
    ax.annotate(label, (x, y), fontsize=10, ha='right', va='bottom', color='blue')

# Plot Illustrative Variables (Red)
if not df_per_activ.empty:
    ax.scatter(df_per_activ[d1], df_per_activ[d2], 
               s=30, alpha=0.8, c='red', marker='s', label='Illustrative Variables', zorder=4)
    
    # Label Illustrative Variables (optional: adjust font size/color to distinguish)
    for label, x, y in zip(df_per_activ.index, df_per_activ[d1], df_per_activ[d2]):
        ax.annotate(label, (x, y), fontsize=10, ha='left', va='top', color='red', fontweight='bold')

# Add reference lines
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')

# Set labels and title
ax.set_xlabel(f"Component 1 ({ax1})") # Formatting percentage if applicable
ax.set_ylabel(f"Component 2 ({ax2})")
ax.set_title("MCA – Axis 1 & 2 (Active vs Illustrative)")

# Add legend to distinguish groups
ax.legend(loc='best')

# Save the figure
img_address='doc_images/mca_prince_with_periods.png'
plt.savefig(img_address, dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

## DBSCAN


Cluster with the DBSCAN method

In [ ]:
# Optional: Scale coordinates (often helps DBSCAN)
scaler = StandardScaler()
mca_scaled = scaler.fit_transform(X_mca)
print(mca_scaled[:2])


### Prepare hyperparameters

In [ ]:


# 2. Define min_samples first
# Rule of thumb: min_samples >= Number of Dimensions + 1
# For MCA, if you kept 5 dimensions, try min_samples = 6 or higher.
# For noisy data, higher is better (e.g., 10-20).
# We take 20, minimum number of individuals in clusters
min_samples_val = 30 

# 3. Fit Nearest Neighbors
# We look for the distance to the (min_samples_val)-th nearest neighbor
nbrs = NearestNeighbors(n_neighbors=min_samples_val).fit(mca_scaled)

# 4. Calculate Distances
distances, indices = nbrs.kneighbors(mca_scaled)

# 5. Extract the k-th distance for each point (last column because sorted)
k_distances = distances[:, -1]

# 6. Sort distances in ascending order
k_distances_sorted = np.sort(k_distances)

# 7. Plot the K-Distance Graph
plt.figure(figsize=(10, 6))
plt.plot(k_distances_sorted)
plt.title(f'K-Distance Graph (k={min_samples_val}) for MCA Coordinates')
plt.xlabel('Points sorted by distance')
plt.ylabel(f'{min_samples_val}-th Nearest Neighbor Distance (eps candidate)')
plt.grid(True)
plt.show()

### Apply DBSCAN

In [ ]:
# Candidate eps from the graph
eps_candidate = 0.8 # 0.9
min_samples_val = 50

db = DBSCAN(eps=eps_candidate, min_samples=min_samples_val).fit(mca_scaled)
labels = db.labels_

# Filter out noise points (-1) for scoring
non_noise_mask = labels != -1
if len(np.unique(labels[non_noise_mask])) > 1:
    score = silhouette_score(mca_scaled[non_noise_mask], labels[non_noise_mask])
    print(f"Silhouette Score (eps={eps_candidate}): {score:.3f}")
else:
    print("Not enough clusters formed to calculate Silhouette Score.")

In [ ]:
# Apply DBSCAN
# eps and min_samples need tuning based on your specific data distribution
# elbow at three
db = DBSCAN(eps=eps_candidate, min_samples=min_samples_val, metric='euclidean')
clusters = db.fit_predict(mca_scaled)

# Add cluster label (number) to original dataframe
df_actives['cluster'] = clusters
df_pm['cluster'] = clusters

cluster_counts=df_actives['cluster'].value_counts()
print(len(cluster_counts))

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    print(cluster_counts)

Commentaire:
* The selected population does not allow for the continuation of this method.